In [1]:
import sys
sys.path.append("..")

In [2]:
import os
import torch
from torch.utils.data import DataLoader

In [3]:
from src.dataloader import BraTSDataset
from src.preprocessing import normalize_image, remap_labels, crop_volume
from src.model import UNet3D
from src.train import DiceLoss  # reuse the DiceLoss class you already wrote

In [4]:
data_dir = r"E:\brain tumor segmentation and analysis using cnn\datasets\BraTS2020_TrainingData\MICCAI_BraTS2020_TrainingData"

In [5]:
def test_preprocess(image, label):
    image = normalize_image(image)
    label = remap_labels(label)
    image, label = crop_volume(image, label, crop_size=(64, 64, 64))
    return image, label

In [6]:
dataset = BraTSDataset(data_dir, transform=test_preprocess)
dataset.patient_dirs = dataset.patient_dirs[:3]

In [7]:
loader = DataLoader(dataset, batch_size=1, shuffle=True)

In [8]:
device = torch.device("cpu")
model = UNet3D(in_channels=4, num_classes=4).to(device)
criterion = DiceLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [9]:
model.train()
for images, labels in loader:
    images, labels = images.to(device), labels.to(device)

    optimizer.zero_grad()
    outputs = model(images)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()


    print(f"Batch loss: {loss.item():.4f}")

Batch loss: 0.8955
Batch loss: 0.8949
Batch loss: 0.8784


In [10]:
print("Test run complete — no crashes.")

Test run complete — no crashes.
